In [0]:
%pip install statsmodels
dbutils.library.restartPython()

# 03: Regression with Controls

Notebook 02 found that paid search sellers activate more often than organic search sellers (54.8% vs. 38.9%). This notebook tests whether that difference holds after controlling for seller characteristics that could explain it:

- **Business segment** (what the seller sells)
- **Lead type** (size and kind of business)
- **Business type** (reseller, manufacturer, other)
- **Signing period** (market conditions when the seller joined)

**Models:**
1. Logistic regression: activation ~ channel + controls
2. Linear regression: log revenue ~ channel + controls (active sellers only)
3. Sensitivity check: adding sales rep as a control

Organic search is the reference channel: every other channel is compared against it.

In [0]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from scipy import stats

df = spark.table("workspace.marts.fact_seller_channel_performance").toPandas()
df["revenue_90d"] = df["revenue_90d"].astype(float)
df["is_active"] = df["is_active_90d"].astype(int)

for col in ["business_segment", "lead_type", "business_type"]:
    df[col] = df[col].fillna("unknown")

segment_counts = df["business_segment"].value_counts()
rare_segments = segment_counts[segment_counts < 15].index
df["segment"] = df["business_segment"].where(
    ~df["business_segment"].isin(rare_segments), "other_segment"
)

df["signing_period"] = pd.to_datetime(df["won_month"]).dt.strftime("%Y-%m")
df["signing_period"] = df["signing_period"].replace({"2017-12": "2018-01", "2018-06": "2018-05"})

df["log_revenue"] = np.log1p(df["revenue_90d"])

print(df["segment"].value_counts(), "\n")
print(df["lead_type"].value_counts(), "\n")
print(df["business_type"].value_counts(), "\n")
print(df["signing_period"].value_counts().sort_index())

In [0]:
channel_term = "C(channel_group, Treatment(reference='organic_search'))"
controls = "C(segment) + C(lead_type) + C(business_type) + C(signing_period)"

logit_base = smf.logit(f"is_active ~ {channel_term}", data=df).fit(disp=False)
logit_ctrl = smf.logit(f"is_active ~ {channel_term} + {controls}", data=df).fit(disp=False, maxiter=200)

def channel_odds_ratios(model, label):
    rows = [name for name in model.params.index if name.startswith("C(channel_group")]
    ci = model.conf_int()
    return pd.DataFrame({
        "channel": [r.split("[T.")[1].rstrip("]") for r in rows],
        f"odds_ratio_{label}": np.exp(model.params[rows]).values,
        f"ci_low_{label}": np.exp(ci.loc[rows, 0]).values,
        f"ci_high_{label}": np.exp(ci.loc[rows, 1]).values,
        f"p_{label}": model.pvalues[rows].values,
    }).set_index("channel")

or_table = channel_odds_ratios(logit_base, "base").join(
    channel_odds_ratios(logit_ctrl, "controlled")
).round(3)

print(f"Sellers in model: {int(logit_ctrl.nobs)}, pseudo R-squared: {logit_ctrl.prsquared:.3f}")
or_table

In [0]:
margeff = logit_ctrl.get_margeff(at="overall", dummy=True).summary_frame()
margeff_channel = margeff[margeff.index.str.startswith("C(channel_group")].round(3)
margeff_channel.index = [i.split("[T.")[1].rstrip("]") for i in margeff_channel.index]

logit_controls_only = smf.logit(f"is_active ~ {controls}", data=df).fit(disp=False, maxiter=200)
lr_stat = 2 * (logit_ctrl.llf - logit_controls_only.llf)
lr_df = logit_ctrl.df_model - logit_controls_only.df_model
lr_p = stats.chi2.sf(lr_stat, lr_df)

print(f"Does channel matter overall, after controls? LR chi-square = {lr_stat:.2f}, df = {int(lr_df)}, p = {lr_p:.4f}\n")
margeff_channel

In [0]:
active = df[df["is_active"] == 1]

ols_ctrl = smf.ols(f"log_revenue ~ {channel_term} + {controls}", data=active).fit(cov_type="HC3")

rows = [n for n in ols_ctrl.params.index if n.startswith("C(channel_group")]
ci = ols_ctrl.conf_int()
revenue_effects = pd.DataFrame({
    "channel": [r.split("[T.")[1].rstrip("]") for r in rows],
    "pct_diff_vs_organic": (np.exp(ols_ctrl.params[rows]) - 1).values * 100,
    "ci_low_pct": (np.exp(ci.loc[rows, 0]) - 1).values * 100,
    "ci_high_pct": (np.exp(ci.loc[rows, 1]) - 1).values * 100,
    "p_value": ols_ctrl.pvalues[rows].values,
}).set_index("channel").round(2)

print(f"Active sellers in model: {int(ols_ctrl.nobs)}, R-squared: {ols_ctrl.rsquared:.3f}\n")
print(ols_ctrl.wald_test_terms(scalar=True), "\n")
revenue_effects

In [0]:
rep_counts = df["sr_id"].value_counts()
df["sales_rep"] = df["sr_id"].where(df["sr_id"].isin(rep_counts[rep_counts >= 10].index), "other_rep")

logit_rep = smf.logit(
    f"is_active ~ {channel_term} + {controls} + C(sales_rep)", data=df
).fit(disp=False, maxiter=300)

comparison = channel_odds_ratios(logit_ctrl, "controlled").join(
    channel_odds_ratios(logit_rep, "with_rep")
)[["odds_ratio_controlled", "p_controlled", "odds_ratio_with_rep", "p_with_rep"]].round(3)

print(f"Sales reps in model: {df['sales_rep'].nunique()}")
comparison

## Results summary

**Activation (logistic regression, 667 sellers)**

| Channel vs. organic search | Odds ratio (no controls) | Odds ratio (with controls) | With sales rep control |
|---|---|---|---|
| paid_search | 1.90 (p = 0.003) | 2.15, 95% CI 1.35 to 3.43 (p = 0.001) | 2.21 (p = 0.001) |
| unknown | 1.07 | 1.05 | 1.00 |
| social | 1.21 | 1.20 | 1.12 |
| direct_traffic | 1.28 | 1.29 | 1.34 |
| other | 0.81 | 0.82 | 0.74 |

- Paid search sellers are **16.7 percentage points more likely to activate** than comparable organic search sellers (95% CI: +6.7 to +26.7 points, p = 0.001), holding business segment, lead type, business type, and signing period constant.
- Channel matters overall after controls (likelihood ratio test: chi-square = 13.84, df = 5, p = 0.017).
- The effect **grows slightly** when controls are added (odds ratio 1.90 to 2.15) and is unchanged when sales rep is added (2.21), so it is not explained by seller mix or by which sales rep handled the deal.
- No other channel differs significantly from organic search.

**Revenue among active sellers (linear regression on log revenue, 289 sellers, robust standard errors)**

- Channel does not predict revenue once a seller is active (joint test p = 0.39). Paid search: +2.4% vs. organic (95% CI: -35% to +61%).
- Lead type does predict revenue (p = 0.037): the kind of business matters more than how it was acquired.

**Limitations:** observational data, so unmeasured differences between paid and organic leads could still explain part of the effect. The models explain a modest share of the variation (pseudo R-squared 0.10 for activation, R-squared 0.17 for revenue).